# Styles and Colors

**Part I · Visualization** — Tutorial 06

Style entities with per-type style dataclasses (`PointStyle`, `LineStyle`,
`PlaneStyle`, `SphereStyle`, `CircleStyle`, …) and the `Color` enum. You will
learn to:

- Apply per-call styles and understand the color/opacity precedence.
- Configure defaults with `viz.styles` (per-kind), `viz.global_styles`, and
  `set_default_color()`.
- Set label defaults (`label_base` / `label_kind`).
- Use wireframe, point size, line thickness, and `CylinderLineStyle`.
- Work with the viz-only entities (`Cylinder`, `Arc`, `Disk`, `PartialDisk`,
  `Box`, `Ellipsoid`, `Ellipse`, `RegularPolygon`).


## Setup


In [ ]:
from pytanga.geometry import (
    Arc, Box, Cylinder, Direction, Disk, Ellipse, Ellipsoid, PartialDisk, Point, Sphere,
    regular_polygon,
)
from pytanga.viz import (
    ArcStyle, BoxStyle, Color, CylinderLineStyle, CylinderStyle, DiskStyle,
    EllipseStyle, EllipsoidStyle, PartialDiskStyle, PointStyle, RegularPolygonStyle,
    SphereStyle, Visualizer,
)


## 1. Per-type style dataclasses and the `Color` enum

Each entity kind has a style class with all fields defaulting to `None` — the
`Visualizer` fills unset fields from its canonical defaults. `Color` is a
string-backed enum, so `Color.RED` is a valid color value.


In [ ]:
print(Color.RED, Color.BLUE, Color.YELLOW)   # '#ff0000' '#0000ff' '#ffff00'

viz = Visualizer(title="Styles — basics", add_default_axes=False, add_default_grid=False)
viz.add(Point(0, 0, 0), color=Color.YELLOW, style=PointStyle(size=0.15), label="origin")
viz.add(Sphere(Point(3, 0, 0), 1.2), style=SphereStyle(color="#ff8844", opacity=0.4, wireframe=True))
viz.display_snapshot()


## 2. Color / opacity precedence

For a given entity, color and opacity resolve in this order (highest first):

1. `add(color=...)` / `add(opacity=...)` shortcut,
2. `style=Style(color=..., opacity=...)`,
3. `viz.styles` per-kind defaults,
4. the canonical built-in defaults.


## 3. Defaults: `viz.styles`, `viz.global_styles`, `set_default_color()`

- `viz.styles` holds the **main scene's** defaults; `viz.global_styles` is the
  **master template** that *new* scenes copy from.
- Assignment (`=`) replaces a whole entry; `merge(...)` overlays only non-`None`
  fields. `set_default_color()` is a quick color-only override (accepts RGBA
  tuples for opacity).


In [ ]:
viz = Visualizer(title="Styles — defaults", add_default_axes=False, add_default_grid=False)

# Mutate canonical defaults via class-based access:
viz.styles[Sphere].wireframe = False
viz.styles[Sphere].opacity = 0.6

# Quick color-only override (hex, RGB, or RGBA tuple):
viz.set_default_color("point", "#00ff00")
viz.set_default_color("sphere", (0.0, 0.0, 1.0, 0.5))  # blue + 50% opacity

# Merge only color (keeps wireframe/opacity), or assign a whole entry:
viz.styles.kind.merge("Sphere", SphereStyle(color="#00ff00"))
# viz.styles["Sphere"] = SphereStyle(color="#00ff00")   # replaces (resets other fields)

viz.add(Point(0, 0, 0), label="green point (default)")
viz.add(Sphere(Point(3, 0, 0), 1.2), label="sphere")
viz.display_snapshot()


## 4. Label defaults

Set global label defaults via `viz.styles.label_base`, and per-kind overrides via
`viz.styles.label_kind` (priority: user `label_style` > per-kind > global).


In [ ]:
from pytanga.viz import LabelStyle

viz = Visualizer(title="Styles — label defaults", add_default_axes=False, add_default_grid=False)

viz.styles.label_base.offset_local = (0.0, 1.1, 0.0)   # all labels sit above their entity
viz.styles.label_kind["Sphere"] = LabelStyle(offset_local=(0.0, 1.05, 0.0))

viz.add(Point(0, 0, 0), color="#ff4444", label="P")
viz.add(Sphere(Point(3, 0, 0), 1.2), opacity=0.4, label="S")
viz.display_snapshot()


## 5. Wireframe, point size, line thickness, and `CylinderLineStyle`

- Wireframe is controlled via `wireframe`, `wireframe_color`, `wireframe_opacity`
  and the `wireframe_dash` pattern (solid / dashed / dotted).
- `PointStyle.size` sets the point marker size.
- `LineStyle.thickness` is a screen-space pixel width; `CylinderLineStyle`
  renders lines as world-unit cylinders instead (use `viz.styles["Line"] = ...`
  to change the default for all lines).

> **Note:** the wireframe-dash classes live in `pytanga.viz._styles` (not
> re-exported from `pytanga.viz`).


In [ ]:
from pytanga.viz import LineStyle
from pytanga.viz._styles import DashedWireframe, DottedWireframe

viz = Visualizer(title="Styles — wireframe & lines", add_default_axes=False, add_default_grid=False)

# Screen-space fat line (pixel width):
viz.add(
    Line(origin=Point(0, 0, 0), direction=Direction(1, 0, 0)),
    color="#44ff44", style=LineStyle(thickness=3.0),
)

# World-unit cylinder line (default for all subsequently-added lines):
viz.styles["Line"] = CylinderLineStyle(thickness=0.04)

# Dashed / dotted wireframe on spheres:
viz.add(Sphere(Point(3, 0, 0), 1.2), style=SphereStyle(
    wireframe=True, wireframe_dash=DashedWireframe(), wireframe_color="#00ffff", opacity=0.3,
))
viz.add(Sphere(Point(6, 0, 0), 1.2), style=SphereStyle(
    wireframe=True, wireframe_dash=DottedWireframe(), wireframe_opacity=0.5, opacity=0.3,
))

viz.display_snapshot()


## 6. Viz-only entities

`Cylinder`, `Arc`, `Disk`, `PartialDisk`, `Box`, `Ellipsoid`, `Ellipse`, and
`RegularPolygon` (via the `regular_polygon()` factory) exist **purely for
rendering** — they have no multivector representation. The flat shapes default
to the xy-plane (`normal = +z`), so they work in both 2D and 3D scenes.


In [ ]:
viz = Visualizer(title="Styles — viz-only entities", add_default_axes=False, add_default_grid=False)

viz.add(Cylinder(origin=Point(0, 0, 0), axis=Direction(0, 0, 1), length=2.0, radius=0.2), color="#44aaff")
viz.add(Arc(origin=Point(2, 0, 0), axis=Direction(0, 0, 1), radius=1.2, tube_radius=0.05), color="#ffcc44")
viz.add(Disk(center=Point(-2, 0, 0), radius=1.0), style=DiskStyle(color="#ff8844", thickness=0.1))
viz.add(PartialDisk(center=Point(0, 2, 0), radius=1.0, angle=4.0), style=PartialDiskStyle(color="#ff44ff", thickness=0.1))
viz.add(Box(center=Point(3, 2, 0), size=(1.2, 1.0, 1.0)), style=BoxStyle(color="#88ccff"))
viz.add(Ellipsoid(center=Point(-3, 2, 0), radii=(1.0, 0.5, 0.75)), style=EllipsoidStyle(color="#ffaa00"))
viz.add(Ellipse(center=Point(0, -2, 0), radius_u=1.2, radius_v=0.6), style=EllipseStyle(color="#ff44ff", thickness=0.1))
viz.add(regular_polygon(6, radius=1.0, center=Point(3, -2, 0)), style=RegularPolygonStyle(color="#44ffaa", thickness=0.1))

viz.display_snapshot()


## 7. Shared knobs

The solid style classes share `color`, `opacity`, `wireframe`,
`wireframe_dash`, `wireframe_color`, and `wireframe_opacity`; the flat shapes
(`Disk` / `PartialDisk` / `Ellipse` / `RegularPolygon`) add a slab `thickness`.


## Visual Examples

A styled collection of entities exported via `export_snapshot()`.


In [ ]:
viz = Visualizer(title="Styles — figure", add_default_axes=False, add_default_grid=False)

viz.styles["Line"] = CylinderLineStyle(thickness=0.03)
viz.set_default_color("point", "#00ff00")

viz.add(Point(0, 0, 0), label="P")
viz.add(Sphere(Point(3, 0, 0), 1.5), style=SphereStyle(wireframe=True, wireframe_color="#66ccff", opacity=0.35))
viz.add(Cylinder(origin=Point(-2, 0, 0), axis=Direction(0, 0, 1), length=2.0, radius=0.25), color="#44aaff")
viz.add(regular_polygon(6, radius=1.0, center=Point(6, 0, 0)), style=RegularPolygonStyle(color="#ffaa00", thickness=0.12))

viz.export_snapshot("_output/06_styles.html", overwrite=True)
print("styles figure exported")


## Summary

| Task | API |
|---|---|
| Per-call style | `add(entity, style=SphereStyle(...))` |
| Color shortcut | `add(entity, color=Color.RED)` / `add(entity, color="#ff4444")` |
| Per-kind default | `viz.styles["Sphere"] = ...` / `viz.styles[Sphere].opacity = ...` |
| Master template | `viz.global_styles["Line"] = ...` |
| Merge (sparse) | `viz.styles.kind.merge("Sphere", SphereStyle(color=...))` |
| Color-only default | `viz.set_default_color("point", "#00ff00")` |
| Label defaults | `viz.styles.label_base` / `viz.styles.label_kind` |
| World-unit lines | `viz.styles["Line"] = CylinderLineStyle(thickness=...)` |
| Wireframe dash | `wireframe_dash=DashedWireframe()` (from `pytanga.viz._styles`) |

**Next:** [07 — Axes, Grid & Camera](../07_axes_grid_camera/).
